In [186]:
import os
from typing import TypedDict, Annotated, List, Literal
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, END, MessagesState
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

In [187]:
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [188]:
llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    max_tokens=500,
    reasoning_format="hidden"
)

In [189]:
class AgentState(MessagesState):
    next_state: str = ""
    current_task: str = ""
    research_res: str = ""
    analysis_res: str = ""
    final_res: str = ""
    task_status: bool = False

In [190]:
from langchain_core.prompts import ChatPromptTemplate
def create_supervisor_chain():
    """Creates the supervisor decision chain"""
    
    supervisor_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a supervisor managing a team of agents:
        
1. Researcher - Gathers information and data
2. Analyst - Analyzes data and provides insights  
3. Writer - Creates reports and summaries

"Rules:
- If there is no research, choose researcher.
- If research exists but no analysis, choose analyst.
- If analysis exists but no report, choose writer.
- If a report exists, choose DONE."

Current state:
- Has research data: {has_research}
- Has analysis: {has_analysis}
- Has report: {has_report}

Respond with ONLY the agent name (researcher/analyst/writer) or 'DONE'.
"""),
        ("human", "{task}")
    ])
    
    return supervisor_prompt | llm

In [191]:
def supervisor_agent(state: AgentState):

    messages = state["messages"]

    current_task = state.get("current_task", "")
    if not current_task:
        current_task = messages[0].content if messages else "No task"

    has_research = bool(state.get("research_res"))
    has_analysis = bool(state.get("analysis_res"))
    has_report = bool(state.get("final_res"))

    if not has_research:
        next_agent = "research"
        supervisor_msg = "📋 Supervisor: Let's start with research. Assigning to Researcher..."

    elif not has_analysis:
        next_agent = "analyst"
        supervisor_msg = "📋 Supervisor: Research done. Time for analysis. Assigning to Analyst..."

    elif not has_report:
        next_agent = "writer"
        supervisor_msg = "📋 Supervisor: Analysis complete. Let's create the report. Assigning to Writer..."

    else:
        next_agent = "end"
        supervisor_msg = "✅ Supervisor: All tasks complete! Great work team."

    return {
        "messages": [AIMessage(content=supervisor_msg)],
        "next_state": next_agent,
        "current_task": current_task
    }

In [192]:
def research_agent(state:AgentState):
    messages = state["messages"]
    task = state.get("current_task","supervisor task")

    research_prompt = f"""As a research specialist, provide comprehensive information about: {task}

    Include:
    1. Key facts and background
    2. Current trends or developments
    3. Important statistics or data points
    4. Notable examples or case studies
    
    Be concise but thorough."""

    research_response = llm.invoke([HumanMessage(content=research_prompt)])
    research_data = research_response.content# Create agent message
    agent_message = f"🔍 Researcher: I've completed the research on '{task}'.\n\nKey findings:\n{research_data[:500]}..."

    return{
        "messages":[AIMessage(content=agent_message)],
        "research_res":research_data,
        "next_state":"supervisor"
    }

In [193]:
def analyst_agent(state:AgentState):
    messages = state["messages"]
    task = state.get("current_task","supervisor task")
    research_data = state.get("research_res","No research data available")

    analyst_prompt = f"""As a data analyst, analyze the following research data for the task: {task}\n\n
    Research Data:\n
    {research_data}\n\n
    Include:\n
    1. Key insights and patterns\n
    2. Strategic implications\n
    3. Potential risks or opportunities\n
    4. Actionable recommendations\n
    \n
    Be objective and analytical."""

    analyst_response = llm.invoke([HumanMessage(content=analyst_prompt)])
    analysis_data = analyst_response.content
    # Create agent message
    agent_message = f"📊 Analyst: I've completed the analysis on '{task}'.\n\nKey insights:\n{analysis_data[:500]}..."

    return{
        "messages":[AIMessage(content=agent_message)],
        "analysis_res":analysis_data,
        "next_state":"supervisor"
    }

In [194]:
def writer_agent(state: AgentState):

    task = state.get("current_task", "supervisor task")
    research_data = state.get("research_res", "No research available")
    analysis_data = state.get("analysis_res", "No analysis for this task")

    writer_prompt = f"""As a professional writer, create an executive report based on:

Task: {task}

Research Findings:
{research_data[:1000]}

Analysis:
{analysis_data[:1000]}

Create a well-structured report with:
1. Executive Summary
2. Key Findings
3. Analysis & Insights
4. Recommendations
5. Conclusion

Keep it professional and concise."""

    response = llm.invoke([HumanMessage(content=writer_prompt)])
    final_data = response.content

    agent_message = "✍️ Writer: Report complete! See below for the full document."

    return {
        "messages": [AIMessage(content=agent_message)],
        "final_res": final_data,
        "task_status": True,
        "next_state": "supervisor"
    }

In [195]:
def router(state:AgentState):
    next_state = state["next_state"]

    if next_state == "end" or state.get("task_status",False):
        return END

    if next_state in ["supervisor","analyst","writer","research"]:
        return next_state
    
    return "supervisor"

In [196]:
# Create workflow
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("supervisor", supervisor_agent)
workflow.add_node("research", research_agent)
workflow.add_node("analyst", analyst_agent)
workflow.add_node("writer", writer_agent)

# Set entry point
workflow.set_entry_point("supervisor")

# Add routing 
for node in ["supervisor", "research", "analyst", "writer"]:
    workflow.add_conditional_edges(
        node,
        router,
        {
            "supervisor": "supervisor",
            "research": "research",
            "analyst": "analyst",
            "writer": "writer",
            END: END
        }
    )

graph=workflow.compile()

In [ ]:
response = graph.invoke({
    "messages": [
        HumanMessage(content="What are the benefits and risks of AI in healthcare?")
    ]
})

print(response["final_res"])